# Stage 2 — Model Training

Loads the Foursquare TIST 2015 dataset, fits the UserProfiler on check-in data,
and saves the trained model to `artifacts/user_profiler.joblib`.

**Prerequisites:** `planit/data/foursquare/` must contain both dataset files.
See `01_prepare_foursquare_data.ipynb` for setup.

In [13]:
FS_POI_FILE     = "dataset_TIST2015_POIs.txt"
FS_CHECKIN_FILE = "dataset_TIST2015_Checkins.txt"

In [14]:
import importlib.util, subprocess, sys

_NEEDED = [
    ("gdown",        "gdown"),
    ("joblib",       "joblib"),
    ("scipy",        "scipy"),
    ("scikit-learn", "sklearn"),
]
for pip_name, import_name in _NEEDED:
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} …")
        subprocess.run([sys.executable, "-m", "pip", "install", pip_name, "-q"], check=True)
    else:
        print(f"{pip_name} already present — skipping")

import os, sys, json, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
print("REPO_ROOT:", REPO_ROOT)
print("Ready.")

gdown already present — skipping
joblib already present — skipping
scipy already present — skipping
scikit-learn already present — skipping
REPO_ROOT: /Users/ahmedabdelsalam/Desktop/PlanIt/planit
Ready.


In [15]:
import zipfile, glob

FS_DIR       = os.path.join(REPO_ROOT, "data", "foursquare")
FS_POI_PATH      = os.path.join(FS_DIR, FS_POI_FILE)
FS_CHECKIN_PATH  = os.path.join(FS_DIR, FS_CHECKIN_FILE)

os.makedirs(FS_DIR, exist_ok=True)

# If the txt files aren't there yet, try to extract from any zip in the same folder
for path, label in [(FS_POI_PATH, FS_POI_FILE), (FS_CHECKIN_PATH, FS_CHECKIN_FILE)]:
    if not os.path.exists(path):
        zips = glob.glob(os.path.join(FS_DIR, "*.zip"))
        if zips:
            print(f"Extracting {label} from {zips[0]} …")
            with zipfile.ZipFile(zips[0]) as z:
                match = next((n for n in z.namelist() if n.endswith(label)), None)
                if match:
                    z.extract(match, FS_DIR)
                    extracted = os.path.join(FS_DIR, match)
                    if extracted != path:
                        os.rename(extracted, path)
                else:
                    print(f"  WARNING: {label} not found in zip")
        else:
            print(f"  {label} not found — download the dataset and place it in {FS_DIR}")

print("POI file exists     :", os.path.exists(FS_POI_PATH))
print("Checkin file exists :", os.path.exists(FS_CHECKIN_PATH))

POI file exists     : True
Checkin file exists : True


In [16]:
from ml.data.preprocess import load_foursquare_pois

places = load_foursquare_pois(FS_POI_PATH)

venue_tag_map: dict[str, list[str]] = {
    p["place_id"]: p["tags"] for p in places if p.get("tags")
}

print(f"Loaded {len(places):,} tagged venues")

from collections import Counter
tag_counts = Counter(t for p in places for t in (p.get("tags") or []))
print("\nTop tags in dataset:")
for tag, count in tag_counts.most_common(10):
    print(f"  {tag:<20} {count:,}")

INFO  Read 3680126 Foursquare venues from /Users/ahmedabdelsalam/Desktop/PlanIt/planit/data/foursquare/dataset_TIST2015_POIs.txt
INFO  Kept 1239166 Foursquare venues after tag filtering


Loaded 1,239,166 tagged venues

Top tags in dataset:
  food_and_drink       759,312
  nightlife            221,901
  quick_visit          183,929
  outdoor              133,851
  wellness             110,724
  shopping             105,863
  cultural             98,382
  scenic               97,207
  upscale              97,164
  budget_friendly      74,370


In [17]:
from ml.data.preprocess import load_foursquare_checkins

visits = load_foursquare_checkins(
    checkin_path=FS_CHECKIN_PATH,
    venue_tag_map=venue_tag_map,
    min_checkins=5,
    max_users=50_000,
)

print(f"\nTotal user-venue visit pairs : {len(visits):,}")
print(f"Unique users                 : {len({v['user_id'] for v in visits}):,}")
print(f"Unique venues                : {len({v['place_id'] for v in visits}):,}")
print(f"\nSample visit record:")
print(visits[0])

INFO  Reading Foursquare check-ins from /Users/ahmedabdelsalam/Desktop/PlanIt/planit/data/foursquare/dataset_TIST2015_Checkins.txt …
INFO  Loaded 5564630 relevant check-ins
INFO  Sampled 50000 / 177124 qualifying users
INFO  Built 934659 user-venue visit pairs (50000 unique users)



Total user-venue visit pairs : 934,659
Unique users                 : 50,000
Unique venues                : 379,462

Sample visit record:
{'user_id': '100000', 'place_id': '4b0587a1f964a520989d22e3', 'visit_count': 1, 'tags': ['shopping']}


In [18]:
from ml.models.user_profiler import UserProfiler

profiler = UserProfiler()
profiler.fit(visits=visits, preferences=[])

ARTIFACT_PATH = os.path.join(REPO_ROOT, "artifacts", "user_profiler.joblib")
profiler.save(ARTIFACT_PATH)
print(f"\nProfiler saved to {ARTIFACT_PATH}")

INFO  Fitting UserProfiler: 934659 visits, 0 preference records
INFO  CF weight=0.80  content weight=0.20
INFO  Fitting complete. Embedding matrix shape: (50000, 64)
INFO  UserProfiler saved to /Users/ahmedabdelsalam/Desktop/PlanIt/planit/artifacts/user_profiler.joblib



Profiler saved to /Users/ahmedabdelsalam/Desktop/PlanIt/planit/artifacts/user_profiler.joblib
